# July 28, 2026 SRBench evaluations

This notebook summarizes the evaluation chain submitted in `submit_jobs.sh`.

## SLURM jobs

- Evolve PySR GT-R²: [548743](out/548743.out)
- Evolve PySR R²: [548744](out/548744.out)
- Evolve PySR GT: [548745](out/548745.out)
- Base PySR: [548746](out/548746.out)
- HPO GT: [548747](out/548747.out)
- HPO R²: [548748](out/548748.out)
- HPO GT-R²: [548749](out/548749.out)

In [ ]:
import json
from pathlib import Path

import pandas as pd

import srbench_results_io as srio

RUNS = Path("runs")
METHODS = [
    # method, SLURM id, ground truth requested, black box requested
    ("HPO R²", 548748, True, True),
    ("HPO GT", 548747, True, True),
    ("HPO GT-R²", 548749, True, True),
    ("Evolve PySR R²", 548744, True, True),
    ("Evolve PySR GT", 548745, False, True),
    ("Evolve PySR GT-R²", 548743, True, True),
    ("Base PySR", 548746, False, True),
]

GT_TOTAL = 133 * 10 * 4
BB_TOTAL = 122 * 10

In [ ]:
def load_ground_truth(run_dir):
    path = run_dir / "srbench_full_results.json"
    if not path.exists():
        return 0, None
    keyed = json.loads(path.read_text()).get("results", {})
    present = [e for e in keyed.values() if e.get("present") and e.get("error") is None]
    solve_rate = sum(bool(e.get("solved")) for e in present) / len(present) if present else None
    return len(present), solve_rate


def load_black_box(run_dir):
    path = run_dir / "srbench_black_box_results.json"
    if not path.exists():
        return 0, None
    datasets = json.loads(path.read_text()).get("datasets", {})
    best_r2 = []
    completed = 0
    for trials in datasets.values():
        completed += len(trials)
        for frontier in trials:
            values = [p["test_r2"] for p in frontier if p.get("test_r2") is not None]
            if values:
                best_r2.append(max(values))
    return completed, sum(best_r2) / len(best_r2) if best_r2 else None


rows = []
for method, job_id, has_gt, has_bb in METHODS:
    gt_done, gt_rate = load_ground_truth(RUNS / str(job_id))
    bb_done, bb_r2 = load_black_box(RUNS / str(job_id))
    done = (gt_done if has_gt else 0) + (bb_done if has_bb else 0)
    total = (GT_TOTAL if has_gt else 0) + (BB_TOTAL if has_bb else 0)
    rows.append({
        "method": method,
        "SLURM": job_id,
        "completed/total": f"{done}/{total}",
        "SRBench solve rate": gt_rate,
        "black-box avg best R²": bb_r2,
    })

results = pd.DataFrame(rows).set_index("method")
results["SRBench solve rate"] = results["SRBench solve rate"].map(
    lambda x: f"{x:.1%}" if pd.notna(x) else "—"
)
results["black-box avg best R²"] = results["black-box avg best R²"].map(
    lambda x: f"{x:.3f}" if pd.notna(x) else "—"
)
results